# Práctica: Inyección de Contexto y "RAG Simplificado"

Ayer aprendimos a hablar con el LLM, pero hoy vamos a darle "memoria" sobre nuestros propios datos. Esta técnica de inyectar nuestros propios documentos en la memoria de trabajo del modelo será el **corazón de vuestro proyecto final**.

Aprenderéis a:
1. Leer un archivo de texto con código Python de forma programática.
2. Construir un "Super-Prompt" enlazando vuestras instrucciones y el texto del documento.
3. Forzar a Gemini a no inventar (alucinar) respuestas si la información no está en el texto.
4. Obligar al modelo a devolver datos estructurados, como diccionarios JSON (imprescindible para APIs).

---

## 1. Configuración Inicial del API

Lo primero de todo, instalemos la librería de nuevo y configuremos la API Key como hicimos ayer.

In [ ]:
# !pip install -q -U google-generativeai
import google.generativeai as genai

# ¡ATENCIÓN! Reemplaza esto por tu API Key real
GOOGLE_API_KEY = "TU_API_KEY_AQUI_ENTRE_COMILLAS"

genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel('gemini-1.5-flash')

---
## 2. Leer un Archivo .txt o .md por Programación
Primero necesitamos crear un archivo simulando la base de datos de nuestra empresa. Lo leeremos con código puro de Python.

In [ ]:
# === SIMULACIÓN ==== 
# Ejecuta esta celda para crear un archivo secreto en la memoria de la máquina (Jupyter).
# Si estuvieras fuera de Jupyter, simplemente crearías un archivo llamado "politicas_empresa.md"
contenido_del_archivo = """
# Políticas Internas de TechNova Inc.

## Trabajo Remoto
- Los empleados pueden trabajar desde casa un máximo de 3 días a la semana.
- Es obligatorio estar online en Slack entre las 10:00 y las 14:00.

## Vacaciones 2026
- Todos tienen 25 días laborables de vacaciones pagadas.
- Durante el mes de Agosto no se pueden tomar más de 2 semanas consecutivas.

## Comidas y Oficina
- El menú de la cantina cambia cada semana. El menú del viernes es siempre "Día de Pizza".
- La oficina cierra a las 20:00 y requiere tarjeta magnética para el acceso de fin de semana.
"""

with open("politicas_empresa.md", "w", encoding="utf-8") as f:
    f.write(contenido_del_archivo)

print("Archivo 'politicas_empresa.md' guardado en disco con éxito!")

---
### Ejercicio 1: Inyección de Contexto en el Prompt

Ahora tienes este archivo en disco, pero el modelo Gemini allá en la nube no lo puede ver. Tienes que leer el archivo y mandárselo **dentro** del String de tu prompt. 

**Tu turno:** 
1. Abre y lee el contenido del archivo `politicas_empresa.md` y coge el string.
2. Crea una variable con una `pregunta_usuario` inventada sobre este texto.
3. Concatena todo usando *f-strings* siguiendo el patrón de teoría para aislar el contexto con `--- INICIO DE INFORMACION ---` e instruyendo a que no invente la respuesta.

In [ ]:
# EJERCICIO 1: "RAG Simplificado" con Petición Única.

# 1. Lee tu archivo guardado
# with open... -> mi_contexto

# 2. Crea la pregunta
# pregunta_usuario = "...?"

# 3. Ensambla el prompt gigante. (Fíjate en las triples comillas de Python y cómo usar {variables})
prompt_con_contexto = f"""
Eres un asistente experto de TechNova Inc. 
Responde usando ESTRICTAMENTE la siguiente información. Si no lo sabes, no lo inventes.

--- INICIO DE LA INFORMACIÓN ---
... aquí debe ir la variable del contexto del archivo ...
--- FIN DE LA INFORMACIÓN ---

Pregunta del usuario: .... aqui debe ir la variable de la pregunta del usuario...
"""

# print(prompt_con_contexto) -> Comprueba que queda con un formato impecable

# 4. Haz la petición a Gemini y comprueba si responde correctamente basándose en el texto.
# ... response = model.generate_content(...)

---
## 3. Controlando el Formato de Salida (Extracción a JSON)

Para vuestro backend de FastAPI tendréis que comunicaros con otros sistemas web. Devolver parrafadas románticas como "¡Claro, aquí tienes la respuesta de las vacaciones: 25 días!" es terrible para una API de frontend.  
A veces, no queremos que el asistente nos responda con texto humano, sino que queremos extraer datos estructurados del contexto en un Json estricto.

### Ejercicio 2: Extracción de Estructuras (JSON)
Vamos a crear otro script que lea nuestro archivo `politicas_empresa.md`. Pero esta vez el prompt que generéis no será para "Charlar" con él, sino para extraer datos y convertirlos de texto libre a un diccionario JSON limpio que programáticamente podamos usar en Python.

**Tu turno:**
Prepara las instrucciones (System Prompts) necesarias para que **obliguen** a Gemini a devolver el número máximo de días de teletrabajo a la semana y los días totales de vacaciones que tiene un empleado en 2026.
El output esperado del modelo debe ser algo así para que sea programable en Python:
```json
{
  "dias_max_teletrabajo": 3,
  "dias_vacaciones_2026": 25
}
```
*Tip: Además del prompt, recuerda usar Temperature=0.0 para extracciones certeras.*

In [ ]:
# EJERCICIO 2: Formato de Salida

import json

# Extrae el texto si no lo tenías..
with open("politicas_empresa.md", "r", encoding="utf-8") as f:
    texto_politicas = f.read()

# TRUCO PRO: Si vas a usar JSON, añade en las instrucciones el esquema que debe devolver con ejemplos concretos.
prompt_extraccion = f"""
  ... (aquí tu System Prompt muy detallado de la tarea y de qué tiene que devolver OBLIGATORIAMENTE JSON)...
  
  -- TEXTO -- 
  {texto_politicas}
"""

generation_config_json = genai.types.GenerationConfig(
    temperature=0.0 # Básico para no alucinar estructuras
)

# En la query puedes pasar el config
response = model.generate_content(prompt_extraccion, generation_config=generation_config_json)

# Vamos a ver si ha obedecido:
texto_respuesta = response.text 
print("La respuesta bruta ha sido:")
print(texto_respuesta)

# (Opcional) Limpia la respuesta por si nos devuelve ```json al principio, etc y usa json.loads().
# if "```json" in texto_respuesta: ...
# mi_diccionario = json.loads(texto_respuesta_limpio)
# print(mi_diccionario['dias_vacaciones_2026'])

---
### Ejercicio 3: Extracción de Listas (JSON array)
Para cerrar la práctica, vamos a pedirle al modelo que extraiga una lista de todas las reglas de la oficina y las devuelva en una estructura de lista (array de JSON).

**Tu turno:**
Prepara las instrucciones para que Gemini devuelva una lista (array en JSON) de todas las reglas presentes bajo la sección "Comidas y Oficina" y "Trabajo Remoto".
El output esperado del modelo debe ser algo así:
```json
[
  "Los empleados pueden trabajar desde casa un máximo de 3 días a la semana",
  "Es obligatorio estar online en Slack entre las 10:00 y las 14:00",
  ...
]
```


In [ ]:
# EJERCICIO 3: Lista de Reglas (JSON Array)

prompt_lista = f"""
  ...(Instrucciones para extraer una lista en JSON plano, sin backticks de markdown ni florituras, o bien extrayendola programáticamente en python con json.loads) ...

  -- TEXTO --
  {texto_politicas}
"""

generation_config_array = genai.types.GenerationConfig(
    temperature=0.0
)

# Ejecutar y verificar
# response_lista = model.generate_content(...)
# print(response_lista.text) 


---
## 4. RAG + Chat

Vamos a unir las dos piezas clave: el **Contexto (RAG)** de hoy con el **Chat (Historial)** que vimos ayer en los conceptos básicos. 

En la API de Gemini 1.5, hay una forma muy potente de configurar esto: al instanciar el modelo Base, podemos pasarle un parámetro oculto llamado `system_instruction`. Si metemos ahí todo nuestro contexto (nuestros documentos y reglas inquebrantables de que no debe alucinar), el modelo asumirá esa información como el "background" que subyace a todo el modelo de chat.

### Ejercicio 4: Chatbot conversacional con tu archivo .md
Completa el script para crear un bucle de terminal (`input()`) donde le puedas pedir al chatbot cosas relacionadas con la empresa, y comprueba probando tú mismo si se acuerda de tu mensaje de la repregunta anterior (y de las políticas de empresa al mismo tiempo).

In [ ]:
# EJERCICIO 4: Bucle de Chat con Contexto (RAG + Historial)

import google.generativeai as genai

# 1. Leemos el documento local (simulando la Extracción/RAG)
with open("politicas_empresa.md", "r", encoding="utf-8") as f:
    documento = f.read()

# 2. Definimos las reglas inquebrantables del chatbot y adjuntamos las políticas
instrucciones_sistema = f"""
Eres el asistente virtual de RRHH de TechNova Inc. 
Responde de forma concisa y amigable basándote EXCLUSIVAMENTE en el siguiente documento.
Si el usuario te pregunta por algo (ej. salarios) que no aparezca en las políticas provistas, NO TE LO INVENTES,
responde educadamente que no tienes acceso a esa información.

--- INICIO DEL DOCUMENTO OFICIAL ---
{documento}
--- FIN DEL DOCUMENTO OFICIAL ---
"""

# 3. A diferencia de un prompt de 1 solo uso, aquí inyectamos nuestro "Super-Prompt"
# como `system_instruction` al declarar nuestro modelo de Google.
# Así, cada vez que crees un chat derivado de este modelo modelo_rag,
# el asistente sabrá qué tiene que hacer por detrás sin decírselo de nuevo.
modelo_rag = genai.GenerativeModel(
    model_name='gemini-1.5-flash',
    system_instruction=instrucciones_sistema
)

# 4. Creamos un nuevo objeto 'Chat' que retiene el historial solo de esta sesión
chat_rrhh = modelo_rag.start_chat()

print("=== Chatbot RRHH TechNova iniciado (Escribe 'salir' para terminar) ===")

# 5. El "Bucle del MVP" - Aquí interaccionarás con el chatbot
while True:
    try:
        # El usuario hace una petición en la terminal
        mensaje = input("Tú: ")
        
        # Condición para cerrar el bucle infinito y finalizar la práctica
        if mensaje.strip().lower() in ['salir', 'exit', 'quit']:
            print("Chat finalizado. ¡Hasta la próxima!")
            break
            
        print(f"\nTú: {mensaje}") # Mostramos el mensaje (porque a veces input en los Notebook es un textbox arriba)
        
        # TODO: Usa `chat_rrhh.send_message(mensaje)` para mandar tu texto y capturar su respuesta
        # respuesta_chat = ...
        
        # TODO: Imprime en consola un string que contenga "🤖 Asistente: " y el texto devuelto:
        # print(...)
        # print("-" * 40)
        
    except Exception as e:
        print(f"Hubo un error: {e}")
        break

# -> CUANDO TENGAS EL CÓDIGO:
# Ejecuta la celda.
# 1º Pregúntale si puedes cogerte diciembre entero de vacaciones. (debería decir que no lo sabe, no está en el doc)
# 2º Pregúntale si puedes ir en Agosto entero (debería decirte que solo 2 semanas máximo según tu .md)
# 3º Hazle una repregunta "Y entonces en navidades?" (debería acordarse de qué se refería "entonces" por el contexto del historial).